# Week 1.4: Prompt Engineering — telling the model what you actually want
### Plain-English edition — no coding required

Your twin is no longer maths on your laptop. It is **powered by a real language model**,
and this notebook is where you learn to steer it.

You will not read about prompting techniques. You will send the same request different
ways and **watch the answers change**.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## Before you start: your free key
This notebook talks to a real model, so it needs a **free Groq API key**.

1. Go to **console.groq.com** and sign up (free, no card needed)
2. Create an API key and copy it
3. Back in Colab, click the **🔑 key icon** in the left sidebar
4. Add a secret named exactly **`GROQ_API_KEY`** and paste your key in
5. Switch **Notebook access** on

> **Never paste your key into a cell or a box.** Anyone who sees your notebook —
> or a screenshot, or a video of your screen — would have your key and your bill.

## The order we will work in
1. **Zero-shot** — just ask, and see what you get
2. **Few-shot** — show two worked examples, and watch the answer snap into shape
3. **Chain of thought** — make it slow down and show its working
4. **Temperature** — the dial between predictable and creative
5. **System instructions** — the standing rules, built last, once you know why they matter

In [ ]:
#@title ▶ Step 1 — Wake up my twin (connect it to the AI) { display-mode: "form" }
#@markdown Nothing to fill in. Press play and I will check your key and say hello to the model.

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

c, k = _client()
if c is not None:
    success("Found your key: " + mask(k))
    note("Only the first and last few characters are ever shown - never the whole key.")
    print("")
    avail = usable_models(c)
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k)
    elif not model:
        problem("Your key works, but no chat model is available to it.")
        note("Check console.groq.com to see what your account has access to.")
    else:
        success("Your twin's brain: " + model)
        if len(avail) > 1:
            note("Other models your key can use: " + ", ".join(m for m in avail if m != model)[:150])
        print("")
        print("Saying hello to the model to be sure it really works...")
        print("")
        reply = llm([{"role": "user", "content": "Say hello in one short sentence."}], temperature=0.0, max_tokens=400)
        if reply:
            success("Connected - the model answered.")
            show("THE MODEL REPLIED", reply)
            print("")
            print("Your twin now has a brain. Everything below sends real messages to it.")
            print("")
            print("So what? Three things made that work: a key that proves who you are,")
            print("a model name, and your message. That is all any AI service needs.")

In [ ]:
#@title ▶ Step 1b — Ask my twin anything { display-mode: "form" }
#@markdown Type whatever you like and press play. This is your twin, live.
my_prompt = "Explain what an LLM is, in two or three simple sentences." #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

if not my_prompt.strip():
    problem("Type something to ask first.")
else:
    show("YOU ASKED", my_prompt.strip())
    print("")
    reply = llm([{"role": "user", "content": my_prompt.strip()}], temperature=0.5)
    if reply:
        show("YOUR TWIN SAID", reply)
        u = globals().get("_LAST_USAGE")
        if u:
            print("")
            note("Tokens - sent: " + str(u.prompt_tokens) + ", received: " + str(u.completion_tokens) +
                 ". That is what you are billed on.")
        print("")
        print("Change the box and press play again. Ask it anything:")
        print("   - explain photosynthesis to a 10-year-old")
        print("   - write two lines of a poem about Accra traffic")
        print("   - what is 17 x 23")
        print("")
        print("So what? Right now it answers like a generic assistant,")
        print("because you have not told it who to be. That is Step 6.")

---
## Part 1: Zero-shot — just ask

**Zero-shot** means asking with words alone: no examples, no format, just the request.

The task we will use throughout: your twin sees what you have just done, and predicts
what you will do next — the same job you built by hand in notebook 1.1, now done by an LLM.

In [ ]:
#@title ▶ Step 2 — Zero-shot: just ask it to predict { display-mode: "form" }
#@markdown What have you just been doing? The twin will guess what comes next.
what_i_just_did = "Finished work, had dinner, now" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

import json as _j
def _unfence(t):
    t = t.strip()
    if t.startswith("```"):
        t = t.split(chr(10), 1)[1] if chr(10) in t else t[3:]
        t = t.rsplit("```", 1)[0]
    return t.strip()
if not what_i_just_did.strip():
    problem("Tell me what you have just been doing first.")
else:
    prompt = "Predict my next activity: " + what_i_just_did.strip()
    show("THE WHOLE PROMPT WE SEND (no examples at all)", prompt)
    print("")
    reply = llm([{"role": "user", "content": prompt}], temperature=0.0)
    if reply:
        show("WHAT CAME BACK", reply)
        print("")
        try:
            _j.loads(_unfence(reply)); tidy = True
        except Exception:
            tidy = False
        if tidy:
            success("That happens to be clean data - but you never asked for it,")
            note("so there is nothing stopping the next answer being a paragraph.")
        else:
            problem("Useful to a human. Useless to a program.")
            note("Your code cannot read this. Which activity did it pick? How sure was it?")
            note("You would have to pick the answer apart with string surgery to find out.")
        print("")
        print("So what? Zero-shot is fine when any sensible answer will do.")
        print("The moment you need the SAME SHAPE every time, it falls apart.")
        print("Step 3 fixes that without changing the question at all.")

---
## Part 2: Few-shot — show, don't tell

Same question. This time you hand the model two **worked examples** first, so it can
see the shape you want. Watch what that does.

The examples are yours to edit — change them and the answer changes with them.

In [ ]:
#@title ▶ Step 3 — Few-shot: show it two examples first { display-mode: "form" }
#@markdown **Example 1** — a situation, and the answer you would want:
example_1_input = "Finished gym, lunch, coffee" #@param {type:"string"}
example_1_output = '{"next":"Work","confidence":0.82}' #@param {type:"string"}
#@markdown **Example 2**:
example_2_input = "Dinner, watched TV" #@param {type:"string"}
example_2_output = '{"next":"Sleep","confidence":0.91}' #@param {type:"string"}
#@markdown **Now your turn** — the same situation you used in Step 2:
what_i_just_did = "Finished work, had dinner, now" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

import json as _j
def _unfence(t):
    t = t.strip()
    if t.startswith("```"):
        t = t.split(chr(10), 1)[1] if chr(10) in t else t[3:]
        t = t.rsplit("```", 1)[0]
    return t.strip()
if not what_i_just_did.strip():
    problem("Tell me what you have just been doing first.")
else:
    nl = chr(10)
    prompt = ("Examples:" + nl +
              "Input: " + example_1_input.strip() + nl +
              "Output: " + example_1_output.strip() + nl + nl +
              "Input: " + example_2_input.strip() + nl +
              "Output: " + example_2_output.strip() + nl + nl +
              "Input: " + what_i_just_did.strip() + nl +
              "Output:")
    print("THESE ARE THE EXAMPLES THE MODEL IS SHOWN:")
    print("")
    print("   " + example_1_input.strip())
    print("      -> " + example_1_output.strip())
    print("   " + example_2_input.strip())
    print("      -> " + example_2_output.strip())
    print("")
    print("And then, with no further explanation, it is asked:")
    print("")
    print("   " + what_i_just_did.strip())
    print("      -> ?")
    print("")
    show("THE WHOLE PROMPT WE SEND", prompt)
    print("")
    reply = llm([{"role": "user", "content": prompt}], temperature=0.0)
    if reply:
        show("WHAT CAME BACK", reply)
        print("")
        try:
            data = _j.loads(_unfence(reply))
            success("Clean data your code can read straight away.")
            print("")
            print("   predicted next activity : " + str(data.get("next", "?")))
            print("   how sure it is          : " + str(data.get("confidence", "?")))
            print("")
            print("Notice you never said 'use JSON'. You never named the fields.")
            print("It copied the shape from your two examples.")
        except Exception:
            problem("Not quite the shape of your examples this time.")
            note("Models drift. Try temperature 0, clearer examples, or a third example.")
        print("")
        print("So what? Compare this with Step 2. Same question, same model.")
        print("Two examples turned a chatty sentence into something a program can use.")
        note("Now edit the examples above - change the fields, add a 'why' - and run it again.")
        note("Whatever shape you show it, it will copy.")

---
## Part 3: Chain of thought — make it slow down

Ask something that needs several steps and a model will often blurt out the first
plausible answer. Four words — **"think step by step"** — change that.

In [ ]:
#@title ▶ Step 4 — Watch "think step by step" change the answer { display-mode: "form" }
#@markdown A question with a tempting wrong answer. Most people blurt out 1 cedi. Is that right?
question = "A football and a water bottle cost 11 cedis together. The football costs 10 cedis more than the bottle. How much does the bottle cost?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

import re as _re
def final_number(text):
    nums = _re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(nums[-1]) if nums else None
DEFAULT_Q = "A football and a water bottle cost 11 cedis together."
print("Sending the SAME question twice - once rushed, once asked to think.")
print("")
fast = llm([{"role": "user", "content": question.strip() + chr(10) +
             "Reply with the final number only. No working."}], temperature=0.0, max_tokens=300)
slow = llm([{"role": "user", "content": question.strip() + chr(10) +
             "Think step by step, then give the final answer."}], temperature=0.0, max_tokens=900)
if fast and slow:
    show("1. RUSHED - answer only", fast)
    print("")
    show("2. ASKED TO THINK - working shown", slow)
    print("")
    a, b = final_number(fast), final_number(slow)
    print("Rushed answer:        " + (str(a) if a is not None else "no number found"))
    print("Step-by-step answer:  " + (str(b) if b is not None else "no number found"))
    print("")
    if question.strip().startswith(DEFAULT_Q):
        right = 0.5
        for label, v in [("Rushed", a), ("Step-by-step", b)]:
            if v is None: continue
            (success if abs(v - right) < 1e-6 else problem)(
                label + ": " + ("correct - 0.50 cedis" if abs(v - right) < 1e-6 else "wrong - the bottle costs 0.50 cedis"))
        print("")
        note("Check: bottle 0.50 + football 10.50 = 11.00, and 10.50 is exactly 10 more than 0.50.")
        note("The tempting answer of 1 cedi makes the football 11 - and the total 12.")
        print("")
    elif a is not None and b is not None:
        (success if abs(a - b) < 1e-6 else problem)(
            "The two answers " + ("agree." if abs(a - b) < 1e-6 else "DISAGREE - read the working to see which is right."))
        print("")
    m = str(globals().get("_MODEL", ""))
    if a is not None and b is not None and abs(a - b) < 1e-6 and any(t in m for t in ("gpt-oss", "qwen", "deepseek", "r1")):
        note("Your model (" + m + ") is a REASONING model: it thinks silently before every")
        note("answer, even when told not to. That is why rushing did not catch it out.")
        note("Older and smaller models get this wrong when rushed. The habit still matters:")
        print("")
    print("So what? Asking for the working does two things. It steadies the model,")
    print("because it reads its own steps as it writes them. And it lets YOU check")
    print("the reasoning - which you can never do with a bare number.")
    note("Try your own tricky question: a percentage, a timetable, a split bill.")

---
## Part 4: The creativity dial

**Temperature** decides how adventurous the model is. Near 0 it picks the safest word
every time. High up, it takes risks. Drag the dial yourself.

In [ ]:
#@title ▶ Step 5 — Turn the creativity dial yourself { display-mode: "form" }
#@markdown Something with many possible good answers:
prompt = "Suggest a name for Ama's football team in three words or fewer." #@param {type:"string"}
#@markdown **Drag the dial.** 0 = always the safest word. 1.5 = wild.
temperature = 0.0 #@param {type:"slider", min:0, max:1.5, step:0.05}
#@markdown Ask the same thing this many times:
how_many_tries = 3 #@param {type:"slider", min:2, max:5, step:1}
#@markdown Also run it at the opposite end of the dial, for comparison:
compare_with_the_other_end = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

def run_at(temp, tries):
    outs = []
    for i in range(int(tries)):
        r = llm([{"role": "user", "content": prompt.strip()}], temperature=float(temp), max_tokens=300)
        if r is None:
            problem("Try " + str(i + 1) + " did not come back. Stopping here.")
            return outs
        outs.append(" ".join(r.split()))
    return outs
def report(temp, outs):
    if not outs: return
    dial = int(round(float(temp) / 1.5 * 20))
    print("TEMPERATURE " + format(float(temp), ".2f") + "   [" + "=" * dial + "o" + "." * (20 - dial) + "]")
    for i, o in enumerate(outs, 1):
        print("   try " + str(i) + ": " + o[:72])
    uniq = len(set(o.lower() for o in outs))
    if uniq == 1:
        print("   -> all " + str(len(outs)) + " answers IDENTICAL")
    elif uniq == len(outs):
        print("   -> every answer DIFFERENT")
    else:
        print("   -> " + str(uniq) + " different answers out of " + str(len(outs)))
    print("")
    return uniq
if not prompt.strip():
    problem("Type something to ask first.")
else:
    print("Asking the same question " + str(int(how_many_tries)) + " times at each setting.")
    print("")
    mine = run_at(temperature, how_many_tries)
    u_mine = report(temperature, mine)
    if compare_with_the_other_end and mine:
        other = 0.0 if float(temperature) > 0.7 else 1.3
        theirs = run_at(other, how_many_tries)
        u_other = report(other, theirs)
        if u_mine and u_other:
            lo, hi = (temperature, other) if float(temperature) < other else (other, temperature)
            lo_u = u_mine if float(temperature) < other else u_other
            hi_u = u_other if float(temperature) < other else u_mine
            print("At " + format(float(lo), ".2f") + " you got " + str(lo_u) + " different answer(s).")
            print("At " + format(float(hi), ".2f") + " you got " + str(hi_u) + " different answer(s).")
            print("")
            if hi_u > lo_u:
                success("That is the dial working: higher temperature, more variety.")
            else:
                note("Not much difference this time - short answers sometimes agree anyway. Try a longer question.")
    if not mine:
        note("No answers came back, so there is nothing to compare yet.")
        note("Fix the red message above and press play again.")
    else:
     print("")
     print("So what? Low temperature for anything that must be consistent:")
     print("extracting data, following a format, answering factual questions.")
     print("High temperature for brainstorming, names, stories.")
     note("A twin answering questions about your life wants LOW temperature. Inventing is not a feature there.")
     note("Try the dial at 0.00, then 0.70, then 1.50, and watch the answers drift.")

---
## Part 5: System instructions — the standing rules

You have left this until last for a good reason. You have now felt what goes wrong
without rules: answers in the wrong shape, the wrong length, the wrong voice.

A **system instruction** is the brief you hand the model before anyone types a word.
It is the most powerful single thing you control.

In [ ]:
#@title ▶ Step 6 — Write my twin's rules and test them live { display-mode: "form" }
#@markdown **ROLE** — who is it?
role = "a digital twin of Ama, a 17-year-old student from Accra" #@param {type:"string"}
#@markdown **KNOWLEDGE** — what may it talk about?
knowledge = "Ama's school subjects, her football training, and her daily routine" #@param {type:"string"}
#@markdown **CONSTRAINTS** — what must it never do?
constraints = "never invent facts about Ama, and never claim to be a real human" #@param {type:"string"}
#@markdown **TONE** — how should it sound?
tone = "friendly and direct, like a classmate" #@param {type:"string"}
#@markdown Now ask your twin something:
my_question = "Who are you, and what did you do on Saturday?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

parts = {"ROLE": role, "KNOWLEDGE": knowledge, "CONSTRAINTS": constraints, "TONE": tone}
empty = [k for k, v in parts.items() if not v.strip()]
if empty:
    problem("These parts are still empty: " + ", ".join(empty))
    note("Each line does a different job. Fill them all in and press play again.")
else:
    brief = chr(10).join(k + ": " + v.strip() for k, v in parts.items())
    globals()["_SYSTEM"] = brief
    show("YOUR SYSTEM INSTRUCTION (sent before every single message)", brief)
    print("")
    answer = llm([{"role": "system", "content": brief},
                  {"role": "user", "content": my_question.strip()}], temperature=0.4)
    if answer:
        show("YOUR TWIN ANSWERS", answer)
        print("")
        print("Now compare it with the SAME question and no rules at all:")
        print("")
        plain = llm([{"role": "user", "content": my_question.strip()}], temperature=0.4)
        if plain:
            show("THE SAME MODEL, WITH NO RULES", plain)
            print("")
            print("So what? Same model, same question, completely different answer.")
            print("You did not retrain anything. You just told it who to be.")
            note("Try changing ROLE to a farmer, a nurse, a teacher - then run this again.")

---
## Part 6: Memory

The model forgets you the instant it replies. A conversation exists only because
**you** send the whole history back every single time.

In [ ]:
#@title ▶ Step 7 — Have a real conversation (run me again and again) { display-mode: "form" }
#@markdown Say something to your twin. Press play, change this box, press play again.
what_i_say = "My name is Ama and I play football on Saturdays." #@param {type:"string"}
#@markdown Tick to wipe the conversation and start fresh:
start_over = False #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

if start_over or "_CHAT" not in globals():
    globals()["_CHAT"] = [{"role": "system", "content": twin_rules()}]
    note("Starting a fresh conversation.")
    print("")
chat = globals()["_CHAT"]
if not what_i_say.strip():
    problem("Type something to say first.")
else:
    chat.append({"role": "user", "content": what_i_say.strip()})
    reply = llm(chat, temperature=0.4)
    if reply is None:
        chat.pop()
    else:
        chat.append({"role": "assistant", "content": reply})
        turns = sum(1 for m in chat if m["role"] == "user")
        print("THE CONVERSATION SO FAR (" + str(turns) + " turn" + ("s" if turns != 1 else "") + "):")
        print("")
        for m in chat:
            if m["role"] == "system": continue
            who = "You " if m["role"] == "user" else "Twin"
            for i, line in enumerate(m["content"].split(chr(10))):
                print(("   " + who + ": " if i == 0 else "         ") + line)
        u = globals().get("_LAST_USAGE")
        if u:
            print("")
            note("This turn sent " + str(u.prompt_tokens) + " tokens - the WHOLE conversation plus the rules, " +
                 "not just your last line. It replied with " + str(u.completion_tokens) + ".")
        print("")
        if turns == 1:
            print("Now change the box above to 'What is my name?' and press play again.")
            print("It will remember - because the history travels with every request.")
        else:
            print("So what? The model has no memory of its own. You are re-sending the")
            print("entire chat every time, and paying for all of it every time.")
            note("Watch the token count climb. That is exactly the problem Week 2 solves.")

### 🎯 Exercise: grade your own prompt, then run it

Everything you have learned, turned into a checklist. Write a prompt, have it marked,
then send it for real.

In [ ]:
#@title ▶ Step 8 — Mark my prompt, then actually send it { display-mode: "form" }
#@markdown Write a prompt you would really use. Try a weak one first, then improve it.
my_prompt = "Tell me about my week" #@param {type:"string"}
#@markdown Send it to the twin as well as marking it?
also_send_it = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    if pick: globals()["_MODEL"] = pick
    return pick
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        problem("Your key works, but I could not find a model it can use.")
        note("Check console.groq.com to see which models your account has access to.")
        return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

p = my_prompt.strip(); low = p.lower()
if len(p) < 5:
    problem("Please write a prompt first.")
else:
    checks = [
      ("Says who the model should be",
       any(k in low for k in ["you are", "act as", "role", "pretend", "as a", "twin"]),
       "Start with who it is: 'You are a digital twin of...'"),
      ("Says what to produce",
       any(k in low for k in ["list", "summar", "explain", "write", "predict", "extract", "compare", "suggest", "plan"]),
       "Name the output: a list, a summary, a plan, an explanation"),
      ("Gives something concrete to work with",
       any(ch.isdigit() for ch in p) or len(p.split()) > 18,
       "Add the real details - dates, numbers, the text to work on"),
      ("Sets a limit or a format",
       any(k in low for k in ["json", "bullet", "sentence", "word", "max", "under", "three", "two", "five", "short", "format", "table"]),
       "Say how long or what shape: 'three bullets', 'one sentence', 'as JSON'"),
      ("Says what NOT to do",
       any(k in low for k in ["do not", "don't", "never", "avoid", "without", "only"]),
       "Add a boundary: 'do not invent anything you were not told'"),
    ]
    passed = sum(1 for _, ok, _ in checks if ok)
    show("YOUR PROMPT", p)
    print("")
    print("Marked against five things that matter:")
    print("")
    for name, ok, _ in checks:
        (success if ok else problem)(("PASS  " if ok else "MISS  ") + name)
    print("")
    print("Score: " + str(passed) + " out of 5")
    misses = [fix for name, ok, fix in checks if not ok]
    if misses:
        print("")
        print("To improve it:")
        for m in misses: print("   - " + m)
    else:
        print("")
        success("Nothing missing. That is a prompt a model can actually follow.")
    if also_send_it:
        print("")
        print("Sending it to your twin now...")
        print("")
        out = llm([{"role": "system", "content": twin_rules()}, {"role": "user", "content": p}], temperature=0.3)
        if out:
            show("WHAT YOUR TWIN MADE OF IT", out)
            print("")
            if passed <= 2:
                print("A low score usually shows up as a vague or invented answer.")
                print("Look closely: did it make anything up to fill the gaps you left?")
            else:
                print("A specific prompt gets a specific answer. Compare this with what")
                print("a one-line version of the same request would have produced.")
    print("")
    print("So what? A vague prompt does not get a vague answer.")
    print("It gets a confident, specific answer to a question you never asked.")

---
## Summary & Key Takeaways

1. **Zero-shot** — fine when any sensible answer will do.
2. **Few-shot** — show two examples and the model copies the shape. The fix for "I need it as data".
3. **Chain of thought** — "think step by step" turns a guess into working you can check.
4. **Temperature** — low for facts and formats, high for brainstorming.
5. **System instructions** — role, knowledge, constraints, tone. Constraints are the line people skip.
6. **No memory** — the whole conversation is re-sent, and re-paid for, every turn.

> Prompting is not a magic phrase. It is writing a brief clear enough that a very fast,
> very literal assistant cannot misread it.

> **Keep your key in Secrets, never in a cell.** A key in a screenshot, a shared
> notebook, or a screen recording is a key someone else can spend.

### Next up
Week 2: cleaning messy text, staying inside the token limit, and giving your twin
real knowledge to draw on.